# Lecture 16: Solving an exam

# Documentation reminder
- [Numpy User Guide](https://docs.scipy.org/doc/numpy/numpy-user-1.17.0.pdf)
- [Numpy Reference Documentation](https://docs.scipy.org/doc/numpy/numpy-ref-1.17.0.pdf)
- [Scipy Reference Guide](https://docs.scipy.org/doc/scipy/scipy-ref-1.4.1.pdf)
- [Matplotlib Documentation](https://matplotlib.org/3.1.3/Matplotlib.pdf)

# Question 1: Strings and files (6p)
You are administrating a public server with many users, and you need to audit the security.
As part of this, you need to look into successfull as well as failed login attempts.

The server prints all such events to a file, so now you want to parse and generate some useful statistics.
All events printed to the file looks like:
```text
[timestamp] - EVENTTYPE: MESSAGE
```
where example of successful and failed logins are:
```text
[2018-05-06T13:35:45] - LOGIN_FAILED: mikael@123.45.67.89
[2018-05-06T14:03:10] - LOGIN_SUCCESS: thomas@10.20.30.40
```
but there are also other type of events, like 
```text
[2018-06-06T15:35:45] - FILE_UPLOAD: "exam_results.txt" mikael@123.45.67.89
```
that we are not interested in. See `events.log` as an example.

## Part A (3p)

Write a function `parse_login_event` that parses login attempts, and returns a tuple containing:

1. Datetime of time stamp
2. Login status (true for success, false for fail)
3. Username
4. IP address (e.g. '123.45.67.89')

If it's not a login event, return `None` (the code should be able to ignore all possible types of events that are not login attempts).

**Example:**
```python
>>> x = parse_login_event('[2018-05-06T13:35:45] - LOGIN_FAILED: mikael@123.45.67.89')
>>> print(x)
(datetime.datetime(2018, 5, 6, 13, 35, 45), False, 'mikael', '123.45.67.89')
>>> y = parse_login_event('[2018-05-06T14:03:10] - LOGIN_SUCCESS: thomas@10.20.30.40')
>>> print(y)
(datetime.datetime(2018, 5, 6, 14, 3, 10), True, 'thomas', '10.20.30.40')
>>> z = parse_login_event('[2018-05-06T15:35:45] - FILE_UPLOAD: "exam_results.txt" mikael@123.45.67.89')
>>> print(z)
None
```

*Hint: Parsing is date + time is easy with `datetime.datetime.strptime` (check the standard library for details)*


In [ ]:
import datetime
import collections

def parse_login_event(event):
    datestr, text = event.split(' - ', maxsplit=1)
    event_type, message = text.split(':', maxsplit=1)
    if event_type == 'LOGIN_SUCCESS':
        success = True
    elif event_type == 'LOGIN_FAILED':
        success = False
    else:
        return None

    d = datetime.datetime.strptime(datestr, '[%Y-%m-%dT%H:%M:%S]')
    user, ip = message.strip().split('@')
    return d, success, user, ip

## Part B (3p)

Write a function `login_stats(filename)` that takes an event log file and prints

- the IP with most failed attempts
- the user with most failed attempts
- a nicely formatted table with hourly breakdown of the number of successful and failed login attempts (you may assume a fixed max width of the table)

\newpage

**Example:**

```text
>>> login_stats('events.log')
Most failed IP: 59.232.36.168 (12 attempts)
Most failed user: glenn (25 attempts)
Hour | Failed | Success
-----|--------|--------
   0 |      4 |       0
   1 |     10 |       0
   2 |      1 |       0
   3 |      3 |       0
   4 |      5 |       0
   5 |      2 |       0
   6 |      1 |       2
   7 |      0 |       2
   8 |      0 |       3
   9 |      2 |       6
  10 |      1 |       6
  11 |      1 |       8
  12 |      0 |       5
  13 |      2 |       5
  14 |      1 |       2
  15 |      0 |       5
  16 |      2 |       2
  17 |      8 |       3
  18 |      8 |       1
  19 |      9 |       0
  20 |      8 |       0
  21 |      9 |       1
  22 |     15 |       2
  23 |      8 |       2
```

*Hint: The `collections` module has a __very__ useful `Counter` class.*


In [ ]:
def login_stats(filename):
    hour_failed = collections.Counter()
    hour_success = collections.Counter()
    ip_failed = collections.Counter()
    user_failed = collections.Counter()

    with open(filename) as f:
        for line in f:
            results = parse_login_event(line)
            if results is not None:
                time, success, user, ip = results
                if success:
                    hour_success[time.hour] += 1
                else:
                    hour_failed[time.hour] += 1
                    user_failed[user] += 1
                    ip_failed[ip] += 1

    print('Most failed IP: {} ({} attempts)'.format(*ip_failed.most_common(1)[0]))
    print('Most failed user: {} ({} attempts)'.format(*user_failed.most_common(1)[0]))

    print('Hour | Failed | Success')
    print('-----|--------|--------')
    for hour in range(0, 24):
        print('{:4} | {:6} | {:7}'.format(hour, hour_failed[hour], hour_success[hour]))

login_stats('events.log')

# Question 2: Numpy, scipy, matplotlib (5p)
As a result of an evaluation we have found that the process we are studying can be described by the function
$y = f (x) = (x - a)^2 + b \sin(x)$ with $a = 2$ and $b = 15$.

To continue our study we need to find the minima and root(s) of this function.

*Tip: Start by plotting the function in the intervall $-10 <= x <= 10$ to get a feel for how it looks.)*

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
a = 2
b = 15

def f(x):
    return (x-a)**2 + b*np.sin(x)

x = np.arange(-10, 10, 0.1)

plt.plot(x, f(x))

### Part A (2p)
Using the function minimize from the SciPy library optimize, find the local minima of the function using the methods:

* Conjugate Gradient (CG)
* Broyden-Fletcher-Goldfarb-Shanno (BFGS)
* Powell (Powell)

and the starting guesses for $x$: 1, 5, 7, 10. Print the resulting x, y values for each combination of method and start guess.

### Part B (1p)
Find the global minima of the function using a global optimization method from the optimize library.
Print the resulting x, y values.

### Part C (1p)
Find the roots ($f (x) = 0$) of the function using a suitable method from the optimize library.

*Hint: You probably need to give different start guesses to find the different roots!*

### Part D (1p)

Using a 2x2 subplot grid, plot the function for $-10 <= x <= 10$ in each subplot.

For each of the local methods in Part A, mark the minima found with a suitable marker in subplots 1-3.
In the fourth subplot, mark the global minima and roots with suitable markers. Make sure to use different markers for
minima and roots.

In all subplots add a title, axis labels etc.

*Hint: Matplotlib manual is extensive. You will likely find what you need in part I.*

In [ ]:
# Part A
methods = ['CG', 'BFGS', 'Powell']
starts = [1, 5, 7, 10]

for i, method in enumerate(methods):
    plt.subplot(2,2,i+1)
    plt.plot(x, f(x))
    plt.xlabel('$x$')
    plt.ylabel('$f(x)$')
    plt.title(method)
    plt.grid()

    print('Using method:', method)

    for start in starts:
        res = optimize.minimize(f, start, method=method, options={'disp': False})
        plt.plot(res.x, f(res.x), 'bo')
        print(' Start guess of', start, 'found x of', res.x, 'with function value', f(res.x))

# Part B
print('Global optimization')
res = optimize.brute(f, [(-5, 5)])
print(' Start guess of', 0, 'found x of', res[0], 'with function value', f(res[0]))

# Part C
plt.subplot(2,2,4)
plt.plot(x, f(x))
plt.title('Global and roots')
plt.xlabel('$x$')
plt.ylabel('$f(x)$')
plt.grid()
plt.plot(res[0], f(res[0]), 'bo')

# Part D
for start in [-3, 0, 3, 5]:
    root = optimize.fsolve(f, start)
    plt.plot(root, f(root), 'rx')
    print ('Starting at x=', start, ', found root at x=', root, 'with function value', f(root))

plt.show()

# Question 3. Designing a class (6p)

A common type of matrices has the block matrix form:

$$ A = \begin{pmatrix} K & G \\ G^T & 0 \end{pmatrix} $$

where $K$ and $G$ in turn are matrices, and $K$ represents a system of equations, and $G$ represents a set of constraints.

## Part A (4p)

Write a class `ConstrainedMatrix` whoose constructor takes the two matrices $K$ and $G$ and represents the $A$ matrix described above. Make sure to only store a single copy of $K$ and $G$ in the class.

Implement the methods

 * `shape()` that returns the rows and columns of $A$ as a tuple
 * `dot(x)` that returns the matrix-vector product $A \cdot x$ as an `ndarray`
 
The matrices `K` and `G` and the vector `x` must also be NumPy `ndarray`s.

#### Hints

- The matrix-vector product is defined as:
$
\begin{aligned}
M \cdot x \equiv \sum_j M_{ij} x_j \, , \quad
M^T \cdot x \equiv \sum_j x_j M_{ji}
\end{aligned}
$
- In the `dot` method use that
$
\begin{aligned}
A \cdot x = \begin{pmatrix} K & G \\ G^T & 0 \end{pmatrix} \cdot \begin{pmatrix} x_k \\ x_g \end{pmatrix} = \begin{pmatrix} K \cdot x_k + G \cdot x_g \\ G^T \cdot x_k \end{pmatrix}
\end{aligned}
$
- The shape of a NumPy `ndarray` is given by the `.shape` property.

In [ ]:
import numpy as np

class ConstrainedMatrix:
    def __init__(self, K, G):
        self.K, self.G = K, G

    def shape(self):
        K_rows = self.K.shape[0]
        G_cols = self.G.shape[1]
        size = K_rows + G_cols
        return size, size

    def dot(self, x):
        """ Computes the matrix-vector product A * x = y """

        pos = self.K.shape[1]
        x_k, x_g = x[:pos], x[pos:]
        
        y_k = np.dot(self.K, x_k) + np.dot(self.G, x_g)
        y_g = np.dot(x_k, self.G)
        
        return np.concatenate((y_k, y_g))

In [ ]:
# Provided test example

K = np.array([
    [1,2,3],
    [4,5,6],
    [7,8,9]])

G = np.array([
    [1, 0],
    [0,-1],
    [-1,1]])

x = np.array([1,0,2,3,0])

A = ConstrainedMatrix(K, G)
print(A.shape()) # should be (5, 5)

y = A.dot(x)
print(y) # should be [10, 16, 22, -1, 2]

Part B (2p)
-----------

You want to add a check that the number of rows and columns in `K` are equal to the number of rows in `G`. 
When this is not the case you want your class to throw an exception. 

Create the custom exception `MatrixSizeMismatch` together with a new class `SafeConstrainedMatrix` that throws the custom exception together with an informative error message if `K` and `G` does not have the same number of rows.

**Also** write a simple test code that catches the exception.

In [ ]:
class MatrixSizeMismatch(Exception):
    pass

class SafeConstrainedMatrix(ConstrainedMatrix):
    def __init__(self, K, G):
        
        K_rows, K_cols = K.shape
        G_rows = G.shape[0]
        
        if K_rows != G_rows or K_cols != G_rows:
            raise MatrixSizeMismatch(
                f"Incompatible matrices: K.shape = {K.shape} while G has {G_rows} rows")
        
        super().__init__(K, G)

In [ ]:
G = np.array([
    [1, 0],
    [0,-1],
    [-1,1]])

K = np.array([
    [1,2],
    [3,4],
    [5,6]])

try:
    A = SafeConstrainedMatrix(K, G)
except MatrixSizeMismatch as e:
    print(e)

# Question 4. Designing a class hierarchy (8p)

Vectors are a critical part of almost any software.

Object Oriented Programming can be used to make special versions of vectors for practical and performance reasons. 

Please do not use NumPy or SciPy for this question. Instead use the `sqrt` funtion from the `math` module.

Part A (0.5p)
-------------

Write a base class `BaseVector` with the abstract methods
- `norm(self)` that should return the vector norm
- `__len__(self)` that should return the length of the vector
- `_getitem(self, index)` that should return the element with index `index`

and the real methods
- `__getitem__(self, index)` that returns `self._getitem(index)` if the index is within the bounds and if not, raises `IndexError`.
- `__str__` that uses the `len` and indexing (getitem) to create a string representation of the vector.

**Hints:**
- The `norm` is defined as $\sqrt{\sum_i a_i^2}$
- The `__len__(self)` method enables  `len(v)` function
- The `__getitem__(self, item)` method enables square bracked indexing `v[index]`

In [ ]:
import abc

class BaseVector(metaclass=abc.ABCMeta):

    @abc.abstractmethod
    def norm(self): pass

    @abc.abstractmethod
    def __len__(self): pass

    @abc.abstractmethod
    def _getitem(self, index): pass
    
    def __getitem__(self, index):
        if index < 0 or index > len(self):
            raise IndexError(f'Index {index} outside range for lenght {len(self)}')
        return self._getitem(index)
        
    def __str__(self):
        return '[' + ', '.join([ str(self[index]) for index in range(len(self)) ]) + ']'
    
    # Part E
    def slice(self, start, end):
        if start < 0 or end < 0 or start > len(self) or end > len(self):
            raise IndexError(f'Slice {start}:{end} outside length {len(self)}')
        return self._slice(start, end)

    @abc.abstractmethod
    def _slice(self, start, end): pass    

Part B (1p)
-----------

Write a subclass `ZeroVector` (a vector of all zeros) whoose constructor that only takes a length.

Add implementations to the `len` operator the `norm` and `x[i]` (getitem) methods.

In [ ]:
class ZeroVector(BaseVector):

    def __init__(self, length):
        self.length = length

    def __len__(self): return self.length
    def _getitem(self, index): return 0
    def norm(self): return 0

    def _slice(self, start, end): return ZeroVector(end - start)

In [ ]:
# Given example code

x = ZeroVector(10)
print(x)
print(x.norm())
print(x[5])

try:
    print(x[15])
except IndexError as e:
    print(e)

Part C (2p)
-----------

Write a subclass `Vector` that is a ordinary vector whoose constructor takes a `list` of values and stores it internally.

Add implementations to the `len` operator the `norm` and `x[i]` (getitem) methods.

In [ ]:
from math import sqrt

class Vector(BaseVector):

    def __init__(self, values):
        self.values = values

    def __len__(self): return len(self.values)
    def _getitem(self, index): return self.values[index]
    def norm(self): return sqrt(sum( (x**2 for x in self.values) ))

    def _slice(self, start, end): return VectorSlice(self, start, end)

In [ ]:
# Given example code

x = Vector([3, 2, 6, 4, 2, 8])
print(x.norm(), 'ref 11.533')

Part D (3p)
-----------

Write a subclass `VectorSlice` that takes a `Vector` and a range in terms of a `start` and `end` index and represents the vector slice without copying any data.

Do this by storing a reference to the original vector, and uses the `start` and `end` range to determine what part of the vector to use.

Add implementations to the `len` operator, the `norm` and `x[i]` (getitem) methods.

In [ ]:
class VectorSlice(BaseVector):
    
    def __init__(self, vector, start, end):
        self.vector = vector
        self.start, self.end = start, end

    def __len__(self): return self.end - self.start
    def _getitem(self, index): return self.vector[self.start + index]
    def norm(self): return sqrt(sum(( self.vector[idx]**2 for idx in range(self.start, self.end) )))

    def _slice(self, start, end):
        return VectorSlice(self.vector, self.start + start, self.start + end)

In [ ]:
x = Vector([1, 3, 5, 7, 9, 11, 13, 15, 17, 19])

y = VectorSlice(x, 3, 6) # should be [7, 9, 11]

print(y) 
print(y.norm(), 'ref 15.843')
print(y[1], 'ref 9')

Part E (1.5p)
-------------

Extend `BaseVector` with
- an abstract method `_slice(self, start, end)` that should return a `VectorSlice`, and
- an explicit method `slice(self, start, end)` that returns `self._slice(self, start, end)` if the slice is within bounds of the vector and if not raises `IndexError`.

Extend `ZeroVector`, `Vector`, and `VectorSlice` with implementations of `_slice(self, start, end)`.
Make use of the `VectorSlice` class when possible.


**Hints:**
- The zero vector is a bit special. The `_slice` method does not have to return a `VectorSlice` type.
- A slice of a slice is still just a slice.

In [ ]:
# Example code

x = Vector([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])
y = x.slice(2, 7) # should be [2, 3, 4, 5, 6]
print(y)
z = y.slice(3, 5) # should be [5, 6]
print(z)
u = ZeroVector(10)
v = u.slice(2, 5) # should contain only zeroes length
print(v)

try:
    print(u.slice(4, 11))
except IndexError as e:
    print(e)